In [3]:
from configuration import MultiAgentConfiguration

cfg = MultiAgentConfiguration(
    supervisor_model        = "gpt-4o-mini",
    researcher_model        = "gpt-4o-mini",
    search_api              = "none",         # turn off Tavily calls for fast tests
    ask_for_clarification   = False,
    include_source_str      = False,
    number_of_queries       = 2,
    mcp_server_config       = None            # disable MCP for now
)

/Users/hieudao/Desktop/untitled_folder/deep_research_project/src/open_deep_research/configuration.py:36: SyntaxWarning: invalid escape sequence '\M'
  DEFAULT_REPORT_STRUCTURE_v2 = """


In [1]:
from dataclasses import asdict
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState
import multi_agent as ma

# 1️⃣  Define your dataclass (unchanged)
cfg = MultiAgentConfiguration(
    supervisor_model="gpt-4.1",
    researcher_model="gpt-4.1",
    search_api="none",
    ask_for_clarification=False,
    include_source_str=False,
    number_of_queries=2,
)

# 2️⃣  Convert to a plain dict that *does* implement .copy()
cfg_dict = asdict(cfg)                # ← or cfg.__dict__

# 3️⃣  Wrap in RunnableConfig (or a bare dict) so LangGraph is happy
run_cfg = RunnableConfig(configurable=cfg_dict)
# run_cfg = {"configurable": cfg_dict}   # also works

# 4️⃣  Build initial state
state: MessagesState = {
    "messages": [HumanMessage(content="We are missing KPIs on last-mile delivery. Build a board deck.")]
}

# 5️⃣  Invoke
final = await ma.graph.ainvoke(state, config=run_cfg)

print(final["final_report"])


ModuleNotFoundError: No module named 'langchain_core'

In [ ]:
final

{'final_report': 'Strategic Analysis and Recommendations for Business Transformation\n\n- [I. Executive Summary](#i-executive-summary)\n- [II. Context & Quantified Problem Statement](#ii-context--quantified-problem-statement)\n- [III. Current State Analysis & Competitive Benchmarking](#iii-current-state-analysis--competitive-benchmarking)\n- [IV. Root Cause Analysis](#iv-root-cause-analysis)\n- [V. Solution Framework & Recommendations](#v-solution-framework--recommendations)\n- [VI. Action Plan, Budget, & ROI Analysis](#vi-action-plan-budget--roi-analysis)\n- [VII. Governance, KPIs, & Risk Management](#vii-governance-kpis--risk-management)\n- [VIII. Conclusion & Strategic Recommendations](#viii-conclusion--strategic-recommendations)\n\n## I. Tóm tắt Điều hành\n\nHội đồng được đề nghị thông qua việc triển khai diện rộng các Chỉ số Hiệu suất Chính (KPIs) một cách mạnh mẽ cho hoạt động giao hàng chặng cuối của công ty. Quyết định này dựa trên kết quả thử nghiệm và các dự báo nội bộ cho th

In [ ]:
import datetime, pathlib, os
import pypandoc

def save_report_md_docx(
    md_text: str,
    out_dir: str | pathlib.Path = "../../reports",
    stem: str | None = None,
    dpi: int = 300,
) -> tuple[pathlib.Path, pathlib.Path]:
    """
    Persist *md_text* to Markdown **and** DOCX, keeping local image refs working.
    
    Parameters
    ----------
    md_text : str
        Full report in Markdown.
    out_dir : str | Path
        Destination folder. Created if missing.
    stem : str | None
        Filename stem. Defaults to `board_report_<YYYY-MM-DD>`.
    dpi : int
        Figure resolution passed to Pandoc (`--dpi`).
    
    Returns
    -------
    md_path, docx_path : tuple[pathlib.Path, pathlib.Path]
    """
    out_dir = pathlib.Path(out_dir).expanduser().resolve()
    out_dir.mkdir(parents=True, exist_ok=True)
    
    stem = stem or f"board_report_{datetime.date.today():%Y-%m-%d}"
    md_path   = out_dir / f"{stem}.md"
    docx_path = md_path.with_suffix(".docx")

    # ── 1  Save Markdown ──────────────────────────────────────────────────────
    md_path.write_text(md_text, encoding="utf-8")

    # ── 2  Build Pandoc resource search path
    #      • `${out_dir}`           → 'images/…'
    #      • `${out_dir.parent}`    → '../reports/images/…'
    resource_search = os.pathsep.join([str(out_dir), str(out_dir.parent)])

    # ── 3  Convert to DOCX ────────────────────────────────────────────────────
    pypandoc.convert_file(
        str(md_path),             # input
        "docx",
        outputfile=str(docx_path),
        extra_args=[
            f"--resource-path={resource_search}",
            f"--dpi={dpi}",
            "--standalone",
        ],
    )

    return md_path, docx_path
md_path, docx_path = save_report_md_docx(final["final_report"])

print("✅ Saved:")
print("•", md_path)
print("•", docx_path)


[WARNING] Could not fetch resource ../reports/images/projected_financial_improvements_by_type.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/kpi_proportion_by_unit_and_type.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/lost_revenue_by_unit_type.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/last_mile_delivery_kpis.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/best_in_class_last_mile_kpis.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/missing_measurements_by_metric.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/factor_type_barriers.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/kpi_measurement_frequency.png: replacing image with description
[WARNING] Could not fetch resource ../repo

✅ Saved:
• /Users/hieudao/Desktop/untitled folder/deep_research_project/reports/board_report_2025-07-09.md
• /Users/hieudao/Desktop/untitled folder/deep_research_project/reports/board_report_2025-07-09.docx


In [2]:
!pip install pypandoc


[notice] A new release of pip is available: 23.3.1 -> 25.1.1
[notice] To update, run: /usr/local/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip


In [ ]:
#!/usr/bin/env python3
import pypandoc
import shutil
import re
import os
import sys

def ensure_pandoc():
    """Ensure pandoc is installed and callable."""
    if not shutil.which("pandoc"):
        sys.stderr.write("✖ Error: 'pandoc' not found. Install it from https://pandoc.org/installing.html\n")
        sys.exit(1)

def strip_yaml_front_matter(md: str) -> str:
    """
    Remove a leading YAML front-matter block (--- … ---).
    Only if it starts right at the top of the file.
    """
    return re.sub(r"^---\s*\n(?:.*?\n)*?---\s*\n", "", md, count=1, flags=re.DOTALL)

def convert_md_to_docx(md_path: str, images_dir: str, output_path: str):
    """
    Convert Markdown to DOCX, disabling metadata parsing,
    pointing at images_dir for resources, and stripping YAML on failure.
    """
    # Read source
    with open(md_path, encoding="utf-8") as f:
        md_text = f.read()

    # Build extra args
    extra_args = [
        f"--resource-path={images_dir}",
        "-f", "markdown-yaml_metadata_block"  # disable YAML metadata parsing
    ]

    try:
        pypandoc.convert_text(
            md_text, to="docx", format="md", outputfile=output_path,
            extra_args=extra_args
        )
    except RuntimeError as e:
        err = str(e)
        if "YAML parse exception" in err:
            sys.stderr.write("⚠ Detected YAML parse error — stripping any front-matter and retrying...\n")
            clean = strip_yaml_front_matter(md_text)
            pypandoc.convert_text(
                clean, to="docx", format="md", outputfile=output_path,
                extra_args=extra_args
            )
        else:
            raise

    print(f"✔ Successfully wrote: {output_path}")

if __name__ == "__main__":
    MD_FILE = "/Users/hieudao/Desktop/untitled_folder/deep_research_project/reports/board_report_2025-07-09.md"
    IMAGES_DIR = "/Users/hieudao/Desktop/untitled_folder/deep_research_project/reports/images/"
    OUTPUT_DOCX = os.path.splitext(MD_FILE)[0] + ".docx"

    ensure_pandoc()
    convert_md_to_docx(MD_FILE, IMAGES_DIR, OUTPUT_DOCX)


[WARNING] Could not fetch resource ../reports/images/attrition_rate_and_replacement_costs.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/employee_attrition_trends.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/average_replacement_cost_bar.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/compensation_structure_by_role_location.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/attrition_by_role_location.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/top_attrition_drivers.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/annual_costs_by_type.png: replacing image with description
[WARNING] Could not fetch resource ../reports/images/retention_programs_cost_and_margin.png: replacing image with description
[WARNING] Could not fetch resource 

✔ Successfully wrote: /Users/hieudao/Desktop/untitled_folder/deep_research_project/reports/board_report_2025-07-09.docx
